### dataset prepare

In [1]:
# load model feature and property of images and split train-test set

import os
import pandas as pd
import scipy.io as scio
from os.path import join as pjoin

basedir = r'/n02dat01/users/qdzhao/THINGS'

outdir_ = pjoin(basedir, r'property_analysis/derivatives/2_pro_relation')
ind = scio.loadmat(pjoin(outdir_, f'index_1854to720.mat'))
ind_1854to720 = ind['index_1854to720'][0].squeeze().tolist()
print(len(ind_1854to720))

/share/home/qdzhao/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.2' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


720


In [2]:
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from os.path import join as pjoin

basedir = r'/n02dat01/users/qdzhao/THINGS'
img_fea_dir = pjoin(basedir, r'THINGS_stimuli/THINGS/derivatives/dnn_feature_maps/TDANN')
img_partitions = os.listdir(img_fea_dir)
img_partitions.sort()
img_fea = []

p = img_partitions[0]
part_dir = os.path.join(img_fea_dir, p)
fea = np.load(part_dir,allow_pickle=True)
fea_ = fea.tolist()
print(fea_.keys())

dict_keys(['layer1.0', 'layer1.1', 'layer2.0', 'layer2.1', 'layer3.0', 'layer3.1', 'layer4.0', 'layer4.1'])


In [3]:
# extract CNN acitvation for each image
# feature extraction was done in model_feature_extraction.ipynb
# use resnet50 layer as feature

import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from os.path import join as pjoin

basedir = r'/n02dat01/users/qdzhao/THINGS'
lys = ['layer1.0','layer1.1','layer2.0','layer2.1','layer3.0','layer3.1','layer4.0','layer4.1']

img_tdann = []
for i in range(len(lys)):
    img_fea_dir = pjoin(basedir, r'THINGS_stimuli/THINGS/derivatives/dnn_feature_maps/TDANN')
    img_partitions = os.listdir(img_fea_dir)
    img_partitions.sort()
    img_fea = []
    for p in img_partitions:
        part_dir = os.path.join(img_fea_dir, p)
        fea = np.load(part_dir,allow_pickle=True)
        fea_ = fea.tolist()
        img_fea.append(fea_[lys[i]].flatten())
    #     img_fea.append(fea_['fc8'])
    img_model_ = np.array(img_fea)
#     img_alex = img_fea_[obj12_df.index]
    img_model_sub = img_model_[ind_1854to720]
    print(img_model_sub.shape) #(720,1000)
    img_tdann.append(img_model_sub)

(720, 200704)
(720, 200704)
(720, 100352)
(720, 100352)
(720, 50176)
(720, 50176)
(720, 25088)
(720, 25088)


In [4]:
# img property pcs
obj_pro_pcs_ = scio.loadmat(pjoin(basedir, r'property_analysis/derivatives/2_pro_relation', f'obj_pro_pcs.mat'))
obj_pro_pcs__ = obj_pro_pcs_['obj_pro_pcs']
obj_pro_pcs = obj_pro_pcs__[ind_1854to720,:]
print(obj_pro_pcs.shape) # (720,5)

(720, 5)


In [5]:
from sklearn.model_selection import train_test_split

outdir = pjoin(basedir, r'property_analysis/data/cnn-som/prop-pcs_tdann')
lys = ['layer1.0','layer1.1','layer2.0','layer2.1','layer3.0','layer3.1','layer4.0','layer4.1']
for i in range(len(lys)):
    outdir_ = pjoin(outdir, lys[i])
    prop_train,prop_test,model_train,model_test = train_test_split(obj_pro_pcs,img_tdann[i],test_size=0.1,random_state=10)
    scio.savemat(pjoin(outdir_, f'img_tdann_720.mat'), {'model_train':model_train,'model_test':model_test}) #(648,4096),(72,4096)
    scio.savemat(pjoin(outdir_, f'obj720_pro_pcs.mat'), {'prop_train':prop_train,'prop_test':prop_test}) #(648,5),(72,5) 